# 05. Refinamiento geométrico orientado a SLAM

En este notebook se estudia una mejora geométrica sencilla sobre la baseline inicial:

- **naive ICP**: registro directo, tal como se planteó en el punto 3.
- **refined ICP**: filtrado previo de la nube hacia una región potencialmente más útil para SLAM, seguido de un ICP multiescala.

El objetivo es comprobar si una modificación geométrica simple permite reducir el error temporal sin introducir un modelo adicional.


In [1]:
from pathlib import Path
import json
import subprocess


In [2]:
ROOT = Path('/home/clara/ml-depth-pro/slam_readiness_nuscenes')
RUN_SUMMARY = ROOT / 'outputs' / 'scene-0061' / 'run_summary.json'
SCRIPT_PATH = ROOT / 'scripts' / 'evaluate_pairwise_registration_refined.py'


## Diferencias con respecto al punto 3

En lugar de registrar toda la nube sin preprocesamiento, se introduce una preparación más orientada a odometría:
- selección de una zona útil en distancias medias,
- recorte de alturas extremas,
- eliminación de puntos excesivamente cercanos al vehículo,
- y aplicación de ICP en varias escalas.

La hipótesis de partida es que este procedimiento puede hacer el registro más estable.


In [3]:
cmd = [
    'python',
    str(SCRIPT_PATH),
    '--run-summary', str(RUN_SUMMARY),
    '--dataroot', '/home/clara/datasets/nuscenes',
    '--version', 'v1.0-mini',
]
print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print('returncode:', result.returncode)


python /home/clara/ml-depth-pro/slam_readiness_nuscenes/scripts/evaluate_pairwise_registration_refined.py --run-summary /home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/run_summary.json --dataroot /home/clara/datasets/nuscenes --version v1.0-mini
/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/pairwise_registration_refined_metrics.json
{
  "scene_name": "scene-0061",
  "num_pairs": 4,
  "pairs": [
    {
      "pair_index": 0,
      "source_token": "ca9a282c9e77460f8360f564131a8af5",
      "target_token": "39586f9d59004284a7114a68825e8eec",
      "dt_s": 0.4998960494995117,
      "gt_relative_translation_m": 4.493198374223651,
      "gt_relative_rotation_deg": 0.41686372685380557,
      "pseudo_naive_fitness": 0.7106527267589715,
      "pseudo_naive_rmse": 0.7483844151631609,
      "pseudo_naive_est_translation_m": 4.906395586625986,
      "pseudo_naive_est_rotation_deg": 0.8138763328437236,
      "pseudo_naive_translation_error_m": 0.9947175184967

In [4]:
metrics_path = ROOT / 'outputs' / 'scene-0061' / 'pairwise_registration_refined_metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print(metrics)
else:
    print('Todavia no existe pairwise_registration_refined_metrics.json. Ejecuta antes la celda del script.')


{'scene_name': 'scene-0061', 'num_pairs': 4, 'pairs': [{'pair_index': 0, 'source_token': 'ca9a282c9e77460f8360f564131a8af5', 'target_token': '39586f9d59004284a7114a68825e8eec', 'dt_s': 0.4998960494995117, 'gt_relative_translation_m': 4.493198374223651, 'gt_relative_rotation_deg': 0.41686372685380557, 'pseudo_naive_fitness': 0.7106527267589715, 'pseudo_naive_rmse': 0.7483844151631609, 'pseudo_naive_est_translation_m': 4.906395586625986, 'pseudo_naive_est_rotation_deg': 0.8138763328437236, 'pseudo_naive_translation_error_m': 0.9947175184967094, 'pseudo_naive_rotation_error_deg': 1.1485142937994566, 'pseudo_refined_fitness': 0.6204540888085192, 'pseudo_refined_rmse': 0.38542159471987686, 'pseudo_refined_est_translation_m': 2.1502384090453006, 'pseudo_refined_est_rotation_deg': 4.1654281489619365, 'pseudo_refined_translation_error_m': 3.9645557035021217, 'pseudo_refined_rotation_error_deg': 4.503972502091582, 'gt_naive_fitness': 0.8782949239343275, 'gt_naive_rmse': 0.5504155153558278, 'gt_

## Criterio de lectura

Las métricas más importantes para esta comparación son:
- `pseudo_naive_translation_error_mean_m`,
- `pseudo_refined_translation_error_mean_m`,
- `pseudo_naive_rotation_error_mean_deg`,
- `pseudo_refined_rotation_error_mean_deg`.

Si la variante refinada reduce estos errores, podrá considerarse que aporta una mejora medible sobre la baseline.
